# Khám phá dữ liệu phát hiện biển số xe Việt Nam
### Báo cáo EDA cho môn Nhập môn khoa học dữ liệu

**Nguồn:** [Vietnamese License Plate Detection – Nhi Huynh](https://www.kaggle.com/datasets/miahuynh04/vietnamese-license-plate-detection). Dataset version 1 là phiên bản quan sát trên trang nguồn ngày 12/10/2026; trang ghi giấy phép MIT, chưa có mô tả dữ liệu.

**Mục tiêu:** đếm chính xác số ảnh (sample), số file nhãn, số ảnh có bounding box hợp lệ; khám phá định dạng, kiểu dữ liệu, kích thước, phân bố lớp và chất lượng dữ liệu; trả lời khả năng thống kê xe máy/ô tô.

**Cách chạy trên Kaggle:** Add Input → chọn dataset trên → Run All. Chỉ cần CPU, không cần Internet/GPU. Kết quả được tính từ file thực tế mỗi lần chạy, không dùng số làm tròn “16.5k files” làm số sample.

**Đơn vị phân tích:** một sample = một file ảnh; một annotation = một dòng nhãn; một ảnh có thể có nhiều đối tượng. Số ảnh, số box, số xe và số xe duy nhất là các đại lượng khác nhau. Notebook không huấn luyện mô hình.

**Mục lục:** 1. Nguồn/cấu trúc • 2. Kiểm kê ảnh • 3. Nhãn YOLO • 4. Tổng quan • 5. Kích thước/định dạng • 6. Lớp/box • 7. Minh họa • 8. Chất lượng ảnh • 9. Trùng lặp/rò rỉ • 10. Xe máy/ô tô • 11. Kết luận và xuất báo cáo.


In [ ]:
from pathlib import Path
from collections import Counter
import os, sys, json, hashlib, math, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display, Markdown, FileLink

SEED = 42
ROOT_OVERRIDE = None  # Nếu chạy local: đường dẫn thư mục chứa images/ và labels/
OUT_OVERRIDE = None
CLASS_NAMES_OVERRIDE = {}  # Chỉ điền khi có tài liệu xác nhận ý nghĩa class_id
VEHICLE_CSV = None  # Tùy chọn: CSV đếm xe thủ công, xem mục 10
plt.rcParams.update({'figure.figsize': (11, 4.5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 90)
OUT = Path(OUT_OVERRIDE or ('/kaggle/working/eda_report' if Path('/kaggle').exists() else 'eda_report'))
OUT.mkdir(parents=True, exist_ok=True)
def show_table(title, df):
    display(Markdown('**'+title+'**'))
    display(df)
def finish_plot(name):
    plt.tight_layout()
    plt.savefig(OUT / (name+'.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
print('Python:', platform.python_version(), '| numpy:', np.__version__, '| pandas:', pd.__version__)


## 1. Xác định thư mục và cấu trúc dữ liệu

Trang nguồn có `License Plate Detection Dataset/images/{train,val,test}` và `labels/{train,val,test}`. Notebook tìm đúng thư mục này thay vì phụ thuộc vào tên mount của Kaggle; hỗ trợ thư mục lồng nhau, khác chữ hoa/thường và symbolic link. Nếu chưa tìm được, cell hiển thị cây thư mục để phân biệt input chưa được gắn với cấu trúc khác dự kiến. File khác như YAML, JSON, CSV được liệt kê để kiểm tra metadata; không thực thi nội dung của chúng.


In [ ]:
def discover_dataset(base, override=None):
    search_base = Path(override) if override else Path(base)
    if not search_base.is_dir():
        raise RuntimeError(f'Thư mục input chưa tồn tại: {search_base}. '
                           'Trên Kaggle, hãy Add Input dataset rồi chạy lại; '
                           'nếu chạy local, điền ROOT_OVERRIDE bằng đường dẫn dữ liệu.')
    candidates, visited, preview, archives = [], set(), [], []
    # Path.rglob có thể bỏ qua symbolic link. os.walk theo link và chặn vòng lặp.
    for current, dirs, filenames in os.walk(search_base, followlinks=True):
        folder = Path(current)
        real = folder.resolve()
        if real in visited:
            dirs[:] = []
            continue
        visited.add(real)
        dirs.sort()
        names = {name.casefold(): name for name in dirs}
        if 'images' in names and 'labels' in names:
            candidates.append((folder, folder/names['images'], folder/names['labels']))
        if len(preview) < 50:
            relative = folder.relative_to(search_base).as_posix()
            preview.append(f'{relative}/: {len(dirs)} thư mục, {len(filenames)} file; '
                           f'thư mục con={dirs[:8]}, file mẫu={sorted(filenames)[:3]}')
        archives.extend(str(folder/f) for f in filenames if Path(f).suffix.lower()=='.zip')
    preferred = [item for item in candidates if
                 'vietnamese-license-plate-detection' in item[0].as_posix().lower()
                 or item[0].name.casefold() == 'license plate detection dataset']
    choices = preferred or candidates
    if len(choices) == 1:
        return choices[0]
    print('Input đang kiểm tra:', search_base)
    print('Các mục ở cấp đầu:', sorted(p.name for p in search_base.iterdir())[:30])
    print('Cây thư mục (tối đa 50 dòng):')
    for line in preview: print('  '+line)
    if archives: print('File ZIP chưa giải nén (nếu có):', archives[:10])
    if not candidates:
        raise RuntimeError('Không thấy thư mục chứa đồng thời images/ và labels/. '
                           'Nếu input trống: Add Input → chọn dataset '
                           'miahuynh04/vietnamese-license-plate-detection. '
                           'Nếu đã có file: kiểm tra cây thư mục in ở trên; '
                           'ROOT_OVERRIDE phải trỏ tới dataset đã giải nén. '
                           'Nếu cấu trúc khác, cần điều chỉnh đường dẫn ảnh/nhãn theo cấu trúc đó.')
    raise RuntimeError('Có nhiều dataset phù hợp: '+str([str(p[0]) for p in choices])+
                       '. Điền ROOT_OVERRIDE bằng đường dẫn dataset muốn phân tích.')

ROOT, IMAGE_DIR, LABEL_DIR = discover_dataset('/kaggle/input', ROOT_OVERRIDE)
print('Thư mục ảnh:', IMAGE_DIR)
print('Thư mục nhãn:', LABEL_DIR)
def dataset_files(parent):
    seen = set()
    for current, dirs, filenames in os.walk(parent, followlinks=True):
        real = Path(current).resolve()
        if real in seen:
            dirs[:] = []
            continue
        seen.add(real)
        dirs.sort()
        for name in sorted(filenames):
            p = Path(current)/name
            if p.is_file(): yield p
files = sorted(dataset_files(ROOT))
inventory = pd.DataFrame([{'relative_path': p.relative_to(ROOT).as_posix(),
    'extension': p.suffix.lower() or '(không đuôi)', 'bytes': p.stat().st_size,
    'group': p.relative_to(ROOT).parts[0]} for p in files])
print('ROOT:', ROOT, '| Tổng file:', len(files), '| Dung lượng MiB:', round(inventory.bytes.sum()/2**20,2))
show_table('Kiểm kê theo thư mục và đuôi file', inventory.groupby(['group','extension']).agg(
    files=('bytes','size'), MiB=('bytes',lambda x: x.sum()/2**20)).round(2))
for top, parent in [('images', IMAGE_DIR), ('labels', LABEL_DIR)]:
    for folder in sorted(p for p in parent.iterdir() if p.is_dir()):
        print(f'{top}/{folder.name}: {sum(1 for p in dataset_files(folder)):,} file')
metadata_files = [p for p in files if p.suffix.lower() in {'.yaml','.yml','.json','.csv','.md'}]
print('Metadata:', [p.relative_to(ROOT).as_posix() for p in metadata_files])


## 2. Đọc toàn bộ ảnh và kiểm kê kiểu dữ liệu

Đếm trên **toàn bộ file ảnh**, kể cả ảnh bị lỗi. Mỗi ảnh được decode để kiểm tra khả năng đọc. Kích thước là kích thước pixel lưu trong file; không xoay theo EXIF vì nhãn cần cùng hệ tọa độ. SHA-256 kiểm tra trùng byte; hash pixel RGB kiểm tra trùng ảnh decode.

Độ sáng, độ tương phản và năng lượng gradient được tính trên thumbnail grayscale (cạnh tối đa 256 px), chỉ phục vụ EDA. Không dùng các chỉ số này làm kết luận tự động rằng ảnh “tốt/xấu”.


In [ ]:
IMAGE_EXT = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff','.gif'}
image_paths = [p for p in files if p.is_relative_to(IMAGE_DIR) and p.suffix.lower() in IMAGE_EXT]
assert image_paths, 'Không tìm thấy ảnh. Kiểm tra ROOT.'
def key_split(p, parent):
    rel = p.relative_to(parent)
    return rel.with_suffix('').as_posix(), rel.parts[0] if len(rel.parts)>1 else 'unspecified'
def scan_image(p):
    key, split = key_split(p, IMAGE_DIR)
    row = dict(image_key=key, split=split, path=str(p), relative_path=p.relative_to(ROOT).as_posix(),
        extension=p.suffix.lower(), bytes=p.stat().st_size, readable=False, error='',
        width=np.nan, height=np.nan, format=None, mode=None, channels=np.nan, dtype=None,
        aspect_ratio=np.nan, megapixels=np.nan, brightness=np.nan, contrast=np.nan,
        gradient_energy=np.nan, sha256=None, pixel_sha256=None, dhash=None, exif_orientation=None)
    try:
        row['sha256'] = hashlib.sha256(p.read_bytes()).hexdigest()
        with Image.open(p) as im:
            im.load()
            a = np.asarray(im)
            w,h = im.size
            rgb = im.convert('RGB')
            thumb = rgb.copy(); thumb.thumbnail((256,256))
            g = np.asarray(thumb.convert('L'), dtype=np.float32)
            grad = (np.mean(np.diff(g,axis=1)**2) if g.shape[1]>1 else 0) +                    (np.mean(np.diff(g,axis=0)**2) if g.shape[0]>1 else 0)
            small = np.asarray(rgb.convert('L').resize((9,8)))
            dhash = np.packbits(small[:,1:] > small[:,:-1]).tobytes().hex()
            pixel_hash = hashlib.sha256(f'{w}x{h}:RGB'.encode()+rgb.tobytes()).hexdigest()
            row.update(readable=True,width=w,height=h,format=im.format,mode=im.mode,
                channels=len(im.getbands()),dtype=str(a.dtype),aspect_ratio=w/h,
                megapixels=w*h/1e6,brightness=float(g.mean()),contrast=float(g.std()),
                gradient_energy=float(grad),pixel_sha256=pixel_hash,dhash=dhash,
                exif_orientation=im.getexif().get(274,1))
    except Exception as e:
        row['error'] = type(e).__name__+': '+str(e)
    return row
images = pd.DataFrame([scan_image(p) for p in tqdm(image_paths, desc='Quét toàn bộ ảnh')])
images['key_unique'] = ~images.image_key.duplicated(keep=False)
readable = images[images.readable].copy()
show_table('5 dòng metadata ảnh', images.head())
print('Ảnh đọc được:', len(readable), '| Lỗi đọc:', (~images.readable).sum())


## 3. Đọc và kiểm tra nhãn

Giả thuyết cần xác minh: nhãn detection YOLO có 5 cột `class_id x_center y_center width height`. Tọa độ được chuẩn hóa theo ảnh. Ví dụ **minh họa**, không phải thống kê dataset: `0 0.50 0.60 0.20 0.10`.

Box pixel: `xmin=(xc-w/2)*W`, `ymin=(yc-h/2)*H`, `xmax=(xc+w/2)*W`, `ymax=(yc+h/2)*H`. Parser giữ lại dòng lỗi/khác định dạng trong bảng audit, không âm thầm sửa hay cắt box. Kiểm tra lớp nguyên không âm, NaN/Inf, kích thước dương, tọa độ và cạnh box trong ảnh (dung sai 1e-6).

**Thiếu file nhãn ≠ nhãn rỗng ≠ nhãn hỏng.** File rỗng có thể là ảnh âm tính hoặc chưa gán nhãn; cần kiểm tra trực quan trước khi kết luận.


In [ ]:
label_paths = [p for p in files if p.is_relative_to(LABEL_DIR) and p.suffix.lower()=='.txt']
for p in label_paths[:3]:
    print(p.relative_to(ROOT), repr(p.read_text(encoding='utf-8-sig',errors='replace')[:400]))
class_names, name_sources = {}, []
for p in metadata_files:
    if p.suffix.lower() in {'.yaml','.yml'}:
        try:
            import yaml
            config = yaml.safe_load(p.read_text(encoding='utf-8-sig'))
            names = config.get('names') if isinstance(config,dict) else None
            mapping = dict(enumerate(names)) if isinstance(names,list) else names
            if isinstance(mapping,dict):
                mapping = {int(k): str(v) for k,v in mapping.items()}
                if class_names and class_names != mapping:
                    print('Cảnh báo: metadata lớp không thống nhất:', p)
                else:
                    class_names.update(mapping); name_sources.append(str(p))
        except Exception as e:
            print('Không đọc được metadata:', p.name, str(e))
class_names.update(CLASS_NAMES_OVERRIDE)
print('Tên lớp có tài liệu:', class_names or 'Không có; giữ tên class_<id>')
print('Nguồn tên lớp:', name_sources)
label_records, annotation_records = [], []
for p in tqdm(label_paths, desc='Kiểm tra nhãn'):
    key, split = key_split(p,LABEL_DIR)
    lines, read_error = [], ''
    try:
        lines = p.read_text(encoding='utf-8-sig').splitlines()
    except Exception as e:
        read_error = str(e)
    nonempty = [(i,s.strip()) for i,s in enumerate(lines,1) if s.strip()]
    for lineno, line in nonempty:
        r = dict(image_key=key,split=split,label_path=str(p),line_number=lineno,raw=line,
                 class_id=np.nan,xc=np.nan,yc=np.nan,bw=np.nan,bh=np.nan,valid=False,issue='')
        tok = line.split()
        issue = []
        if len(tok)!=5:
            issue.append(f'expected_5_fields_got_{len(tok)}')
        else:
            try:
                cid,xc,yc,bw,bh = map(float,tok)
                vals = [cid,xc,yc,bw,bh]
                if not np.isfinite(vals).all():
                    issue.append('non_finite')
                else:
                    r.update(class_id=cid,xc=xc,yc=yc,bw=bw,bh=bh)
                    if cid<0 or cid!=int(cid): issue.append('invalid_class_id')
                    if class_names and int(cid) not in class_names: issue.append('unknown_class_id')
                    if not (0<=xc<=1 and 0<=yc<=1 and 0<bw<=1 and 0<bh<=1):
                        issue.append('invalid_normalized_coordinate')
                    if min(xc-bw/2,yc-bh/2)<-1e-6 or max(xc+bw/2,yc+bh/2)>1+1e-6:
                        issue.append('box_outside_image')
            except ValueError:
                issue.append('not_numeric')
        r.update(valid=not issue,issue=';'.join(issue))
        annotation_records.append(r)
    label_records.append(dict(image_key=key,split=split,label_path=str(p),label_bytes=p.stat().st_size,
                              nonempty_lines=len(nonempty),read_error=read_error))
labels = pd.DataFrame(label_records, columns=['image_key','split','label_path','label_bytes','nonempty_lines','read_error'])
ann = pd.DataFrame(annotation_records, columns=['image_key','split','label_path','line_number','raw',
                                               'class_id','xc','yc','bw','bh','valid','issue'])
ann['valid'] = ann.valid.astype(bool)
valid_counts = ann[ann.valid].groupby('image_key').size()
invalid_counts = ann[~ann.valid].groupby('image_key').size()
labels['valid_boxes'] = labels.image_key.map(valid_counts).fillna(0).astype(int)
labels['invalid_lines'] = labels.image_key.map(invalid_counts).fillna(0).astype(int)
labels['status'] = np.select([labels.read_error.ne(''),labels.nonempty_lines.eq(0),labels.invalid_lines.gt(0)],
                            ['unreadable','empty','has_invalid_rows'],default='valid')
labels['n_image_matches'] = labels.image_key.map(images.image_key.value_counts()).fillna(0).astype(int)
orphans = labels[labels.n_image_matches.eq(0)].copy()
images = images.merge(labels.drop(columns='split'),on='image_key',how='left',validate='many_to_one')
images['label_status'] = images.status.fillna('missing')
images.loc[~images.key_unique,'label_status'] = 'ambiguous_image_key'
valid_ann = ann[ann.valid].copy()
unique_images = images[images.key_unique]
boxes = valid_ann.merge(unique_images[['image_key','path','width','height','readable']],
                        on='image_key',how='inner',validate='many_to_one')
boxes['class_id'] = boxes.class_id.astype(int)
boxes['class_name'] = boxes.class_id.map(lambda c: class_names.get(c,f'class_{c}'))
boxes['usable'] = boxes.readable
for col, expr in {'xmin':(boxes.xc-boxes.bw/2)*boxes.width,'xmax':(boxes.xc+boxes.bw/2)*boxes.width,
                  'ymin':(boxes.yc-boxes.bh/2)*boxes.height,'ymax':(boxes.yc+boxes.bh/2)*boxes.height,
                  'width_px':boxes.bw*boxes.width,'height_px':boxes.bh*boxes.height,
                  'area_ratio':boxes.bw*boxes.bh}.items(): boxes[col]=expr
boxes['box_aspect'] = boxes.width_px/boxes.height_px
usable_boxes = boxes[boxes.usable].copy()
show_table('Các dòng nhãn lỗi / khác định dạng (tối đa 20)', ann[~ann.valid].head(20))
show_table('Nhãn mồ côi (không có ảnh cùng split + tên file)', orphans.head(20))


## 4. Tổng quan: bao nhiêu sample, bao nhiêu có nhãn?

“Có file nhãn” là kiểm tra ghép cặp file. “Có box hợp lệ” đòi hỏi ảnh ghép duy nhất và ít nhất một dòng YOLO hợp lệ. “Sẵn sàng sử dụng” còn đòi hỏi ảnh đọc được và toàn bộ file nhãn không có dòng lỗi. Nhãn rỗng chưa được mặc định xem là mẫu âm tính đã xác minh.


In [ ]:
images['has_valid_box'] = images.key_unique & images.valid_boxes.fillna(0).gt(0)
images['ready_positive'] = images.readable & images.key_unique & images.label_status.eq('valid') & images.has_valid_box
rows=[]
for split, group in images.groupby('split',sort=True):
    rows.append(dict(split=split,samples=len(group),readable=int(group.readable.sum()),
        label_file=int(group.label_path.notna().sum()),missing_label=int(group.label_status.eq('missing').sum()),
        empty_label=int(group.label_status.eq('empty').sum()),
        invalid_label=int(group.label_status.eq('has_invalid_rows').sum()),
        unreadable_label=int(group.label_status.eq('unreadable').sum()),
        ambiguous_key=int((~group.key_unique).sum()),has_valid_box=int(group.has_valid_box.sum()),
        ready_positive=int(group.ready_positive.sum()),
        valid_boxes=int(boxes.split.eq(split).sum())))
split_summary=pd.DataFrame(rows).set_index('split')
split_summary['sample_percent']=100*split_summary.samples/len(images)
show_table('Bảng tổng quan theo split',split_summary.round(2))
headline=pd.DataFrame({'Chỉ tiêu':['Tổng sample (file ảnh)','Ảnh đọc được','Ảnh có file nhãn',
    'Ảnh ghép duy nhất có ≥1 box hợp lệ','Ảnh dương tính sẵn sàng sử dụng','File nhãn rỗng',
    'Ảnh thiếu nhãn','Tổng dòng annotation','Dòng annotation hợp lệ (kể cả mồ côi)',
    'Box hợp lệ ghép được ảnh duy nhất','Box hợp lệ + ảnh đọc được','Dòng annotation lỗi',
    'File nhãn mồ côi','Số class ID quan sát trên dòng hợp lệ'],
    'Giá trị':[len(images),int(images.readable.sum()),int(images.label_path.notna().sum()),
    int(images.has_valid_box.sum()),int(images.ready_positive.sum()),int(labels.status.eq('empty').sum()),
    int(images.label_status.eq('missing').sum()),len(ann),int(ann.valid.sum()),len(boxes),len(usable_boxes),
    int((~ann.valid).sum()),len(orphans),int(ann.loc[ann.valid,'class_id'].nunique())]})
show_table('Các con số để dùng trong báo cáo',headline)
fig,axs=plt.subplots(1,2,figsize=(13,4))
split_summary.samples.plot.bar(ax=axs[0],color='#2563eb',rot=0,title='Số sample theo split',ylabel='Ảnh')
for i,v in enumerate(split_summary.samples): axs[0].text(i,v,str(v),ha='center',va='bottom')
pd.crosstab(images.split,images.label_status).plot.bar(stacked=True,ax=axs[1],rot=0,
    title='Trạng thái file nhãn',ylabel='Ảnh')
axs[1].legend(title='Trạng thái',bbox_to_anchor=(1,1))
finish_plot('01_overview')


## 5. Định dạng, kiểu dữ liệu và kích thước ảnh

Đuôi `.jpg` và định dạng decode thực tế được thống kê riêng. `mode` cho biết RGB, grayscale, palette…; `dtype` là kiểu mảng gốc (thường uint8); các chỉ số EDA sử dụng ảnh chuyển RGB/grayscale. Tất cả thống kê kích thước chỉ dùng ảnh đọc được; đơn vị chiều rộng/cao là pixel, dung lượng là KiB/MiB.


In [ ]:
readable=images[images.readable].copy()
show_table('Kiểu dữ liệu của bảng EDA',images.dtypes.astype(str).to_frame('dtype'))
show_table('Định dạng file / mode / số kênh / dtype',readable.groupby(
    ['extension','format','mode','channels','dtype'],dropna=False).size().to_frame('n_images'))
size_stats=readable[['width','height','aspect_ratio','megapixels','bytes']].describe(
    percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
show_table('Thống kê kích thước (bytes là dung lượng file)',size_stats.round(3))
show_table('10 kích thước phổ biến nhất',readable.groupby(['width','height']).size().sort_values(
    ascending=False).head(10).to_frame('n_images'))
show_table('Kích thước theo split',readable.groupby('split').agg(
    width_min=('width','min'),width_median=('width','median'),width_max=('width','max'),
    height_min=('height','min'),height_median=('height','median'),height_max=('height','max')))
fig,axs=plt.subplots(2,2,figsize=(13,9))
for ax,col,title in zip(axs.flat,['width','height','aspect_ratio','bytes'],
                       ['Chiều rộng (px)','Chiều cao (px)','Tỉ lệ rộng/cao','Dung lượng (KiB)']):
    x=readable[col]/1024 if col=='bytes' else readable[col]
    ax.hist(x.dropna(),bins=40,color='#0d9488',edgecolor='white'); ax.set(xlabel=title,ylabel='Ảnh')
finish_plot('02_image_dimensions')
plt.scatter(readable.width,readable.height,s=10,alpha=.25)
plt.xlabel('Chiều rộng (px)');plt.ylabel('Chiều cao (px)');plt.title('Quan hệ chiều rộng và chiều cao')
finish_plot('03_width_height')


## 6. Phân bố lớp và đặc điểm bounding box

Các biểu đồ dưới dùng box hợp lệ ghép được với ảnh đọc được. Khi không có tên lớp được xác minh, `class_0` chỉ là mã lớp; chưa đủ để khẳng định đó là xe máy, ô tô hoặc biển số một/hai dòng.

Phân tích kích thước box gồm kích thước pixel, diện tích tương đối, vị trí tâm và số box/ảnh. Box cạnh ngắn <16 px là **ngưỡng khảo sát** để tìm đối tượng nhỏ, không phải tiêu chuẩn chất lượng phổ quát. Các box trùng annotation vẫn được đếm ở thống kê gốc và có bảng audit riêng.


In [ ]:
class_summary=usable_boxes.groupby(['class_id','class_name']).agg(
    n_boxes=('image_key','size'),n_images=('image_key','nunique')).reset_index()
show_table('Lớp quan sát (số ảnh giữa các lớp có thể chồng lấp)',class_summary)
show_table('Box theo split và lớp',pd.crosstab(usable_boxes.split,usable_boxes.class_name))
box_stats=usable_boxes[['width_px','height_px','box_aspect','area_ratio']].describe().T
show_table('Đặc điểm box hợp lệ',box_stats.round(4))
box_per_image=usable_boxes.groupby('image_key').size()
ready=images[images.readable & images.key_unique & images.label_status.isin(['valid','empty'])].copy()
ready['n_boxes']=ready.image_key.map(box_per_image).fillna(0).astype(int)
show_table('Phân bố số box/ảnh có nhãn đọc được, không lỗi',ready.n_boxes.value_counts().sort_index().to_frame('n_images'))
duplicate_annotations=usable_boxes[usable_boxes.duplicated(['image_key','class_id','xc','yc','bw','bh'],keep=False)]
print('Dòng thuộc nhóm annotation trùng chính xác:',len(duplicate_annotations))
print('Box có cạnh ngắn <16 px:',int((usable_boxes[['width_px','height_px']].min(axis=1)<16).sum()),'/',len(usable_boxes))
if len(usable_boxes):
    fig,axs=plt.subplots(2,2,figsize=(13,9))
    class_summary.set_index('class_name').n_boxes.plot.bar(ax=axs[0,0],rot=20,color='#7c3aed',ylabel='Box',title='Phân bố lớp')
    axs[0,1].hist(usable_boxes.area_ratio*100,bins=40,color='#2563eb');axs[0,1].set(xlabel='Diện tích box / ảnh (%)',ylabel='Box')
    axs[1,0].hist(usable_boxes.box_aspect,bins=40,color='#0d9488');axs[1,0].set(xlabel='Tỉ lệ rộng/cao của box',ylabel='Box')
    h=axs[1,1].hist2d(usable_boxes.xc,usable_boxes.yc,bins=25,range=[[0,1],[0,1]],cmap='Blues')
    axs[1,1].invert_yaxis();axs[1,1].set(xlabel='Tâm x chuẩn hóa',ylabel='Tâm y chuẩn hóa',title='Vị trí tâm box')
    fig.colorbar(h[3],ax=axs[1,1],label='Số box')
    finish_plot('04_box_distributions')


## 7. Minh họa ảnh và nhãn

Lấy mẫu tái lập với seed 42 từ từng split. Khung chỉ hiển thị box hợp lệ; ảnh thiếu/rỗng/lỗi nhãn hiển thị trạng thái rõ ràng. Có thêm crop đối tượng nhỏ và ảnh có nhiều box để kiểm tra annotation bằng mắt. Lưới mẫu không thay thế việc kiểm tra toàn bộ dữ liệu.


In [ ]:
def image_grid(frame,title,name,n=9,cols=3):
    frame=frame[frame.readable].head(n)
    if frame.empty:
        print(title,': không có ảnh phù hợp');return
    fig,axs=plt.subplots(math.ceil(len(frame)/cols),cols,figsize=(cols*4,math.ceil(len(frame)/cols)*3.5),squeeze=False)
    for ax in axs.flat: ax.axis('off')
    for ax,(_,r) in zip(axs.flat,frame.iterrows()):
        with Image.open(r.path) as im: ax.imshow(im.convert('RGB'))
        selected=usable_boxes[usable_boxes.image_key.eq(r.image_key)] if r.key_unique else usable_boxes.iloc[:0]
        for _,b in selected.iterrows():
            ax.add_patch(Rectangle((b.xmin,b.ymin),b.width_px,b.height_px,fill=False,edgecolor='#facc15',linewidth=2))
            ax.text(b.xmin,b.ymin,b.class_name,color='white',fontsize=8,
                    bbox={'facecolor':'black','alpha':.6,'pad':1})
        ax.set_title(f'{r.split} | {int(r.width)}×{int(r.height)} | {r.label_status}\n{Path(r.path).name[:40]}',fontsize=8)
    fig.suptitle(title)
    finish_plot(name)
for split,g in images.groupby('split'):
    good=g[g.readable]
    image_grid(good.sample(min(6,len(good)),random_state=SEED),'Ảnh mẫu: '+split,'05_samples_'+split,n=6)
if len(usable_boxes):
    multi=box_per_image.sort_values(ascending=False).head(6).index
    image_grid(images[images.image_key.isin(multi)],'Ảnh có nhiều box nhất','06_multiple_boxes',n=6)
    small_boxes=usable_boxes.sort_values('area_ratio').head(9)
    fig,axs=plt.subplots(3,3,figsize=(12,8))
    for ax in axs.flat: ax.axis('off')
    for ax,(_,b) in zip(axs.flat,small_boxes.iterrows()):
        with Image.open(b.path) as im:
            bounds=(max(0,math.floor(b.xmin)),max(0,math.floor(b.ymin)),
                    min(im.width,math.ceil(b.xmax)),min(im.height,math.ceil(b.ymax)))
            ax.imshow(im.convert('RGB').crop(bounds))
        ax.set_title(f'{b.class_name} | {b.width_px:.1f}×{b.height_px:.1f}px',fontsize=9)
    fig.suptitle('Crop 9 box có diện tích tương đối nhỏ nhất (đã phóng to)')
    finish_plot('07_small_crops')


## 8. Chất lượng dữ liệu và các trường hợp cần kiểm tra

Độ sáng grayscale trong [0,255], tương phản là độ lệch chuẩn grayscale; năng lượng gradient phụ thuộc chi tiết ảnh và cách resize. Chỉ chọn các cực trị để review, không tự động loại ảnh. EXIF orientation khác 1 cần xác minh cách công cụ gán nhãn xử lý xoay ảnh.


In [ ]:
quality_summary=pd.DataFrame({'Vấn đề':['Ảnh không đọc được','Ảnh thiếu nhãn','File nhãn rỗng',
    'File nhãn có dòng lỗi','File nhãn không đọc được','Nhãn mồ côi',
    'Ảnh có khóa ghép trùng','Dòng thuộc nhóm annotation trùng','Ảnh EXIF orientation khác 1'],
    'Số lượng':[int((~images.readable).sum()),int(images.label_status.eq('missing').sum()),
    int(labels.status.eq('empty').sum()),int(labels.status.eq('has_invalid_rows').sum()),
    int(labels.status.eq('unreadable').sum()),len(orphans),int((~images.key_unique).sum()),
    len(duplicate_annotations),int((readable.exif_orientation.fillna(1)!=1).sum())]})
show_table('Bảng audit chất lượng (đơn vị ghi trong tên vấn đề)',quality_summary)
show_table('Ảnh lỗi đọc',images[~images.readable][['relative_path','error']].head(20))
show_table('Chỉ số ảnh thumbnail theo split',readable.groupby('split')[['brightness','contrast','gradient_energy']].agg(['mean','median','std']).round(2))
fig,axs=plt.subplots(1,3,figsize=(15,4))
for ax,col in zip(axs,['brightness','contrast','gradient_energy']):
    for split,g in readable.groupby('split'):
        ax.hist(g[col],bins=35,alpha=.4,label=split,density=True)
    ax.set(xlabel=col,ylabel='Mật độ');ax.legend()
finish_plot('08_image_quality')
image_grid(readable.nsmallest(6,'brightness'),'6 ảnh tối nhất theo thumbnail','09_dark_examples',n=6)
image_grid(readable.nsmallest(6,'gradient_energy'),'6 ảnh ít gradient nhất: cần kiểm tra trực quan','10_low_gradient',n=6)
image_grid(images[images.label_status.isin(['missing','empty','has_invalid_rows'])],
           'Ảnh thiếu/rỗng/lỗi nhãn để review','11_label_issues',n=6)


## 9. Trùng lặp và rò rỉ giữa train / val / test

Trùng byte được xác định bằng SHA-256; trùng pixel RGB giúp phát hiện ảnh cùng nội dung decode dù metadata/encoding khác. Nhóm có nhiều split là nguy cơ rò rỉ khi đánh giá mô hình. Số nhóm, số file trong nhóm và số bản sao dư là ba số khác nhau.

Collision dHash là **ứng viên ảnh gần giống**, cần kiểm tra bằng mắt. Chỉ dò dHash giống hoàn toàn trong notebook này; không kiểm tra mọi khoảng cách Hamming và không chứng minh dữ liệu đã hết ảnh gần trùng, cùng xe hay cùng video.


In [ ]:
def duplicate_groups(column):
    table=images.dropna(subset=[column]).groupby(column).agg(
        n_files=('path','size'),n_splits=('split','nunique'),
        splits=('split',lambda x: '|'.join(sorted(set(x)))),
        paths=('relative_path',lambda x: json.dumps(list(x),ensure_ascii=False)))
    return table[table.n_files>1].reset_index()
byte_dups=duplicate_groups('sha256')
pixel_dups=duplicate_groups('pixel_sha256')
dhash_candidates=duplicate_groups('dhash')
duplicate_summary=pd.DataFrame([dict(method=method,groups=len(g),files_in_groups=int(g.n_files.sum()),
    redundant_copies=int((g.n_files-1).sum()),cross_split_groups=int(g.n_splits.gt(1).sum()))
    for method,g in [('SHA256 byte',byte_dups),('SHA256 pixel RGB',pixel_dups),('dHash collision (ứng viên)',dhash_candidates)]])
show_table('Tổng hợp trùng lặp',duplicate_summary)
show_table('Nhóm trùng pixel giữa các split',pixel_dups[pixel_dups.n_splits.gt(1)].head(20))
cross=pixel_dups[pixel_dups.n_splits.gt(1)]
if len(cross):
    image_grid(images[images.pixel_sha256.eq(cross.iloc[0].pixel_sha256)],
               'Ví dụ cùng pixel xuất hiện ở nhiều split','12_cross_split_duplicate',n=6)


## 10. Bao nhiêu xe máy, bao nhiêu ô tô?

**Không tự suy loại xe từ class ID hay tỉ lệ rộng/cao của biển số.** Nếu dataset chỉ gán nhãn vùng biển số thì file nhãn không cung cấp số xe máy/ô tô. Hình dạng box và một vài ảnh mẫu không đủ để đưa ra số chính xác cho toàn dataset.

Để trả lời có kiểm chứng, bổ sung CSV kiểm đếm thủ công: một dòng/mỗi file ảnh với `relative_path,n_motorcycles,n_cars,n_other_vehicles,reviewed`. Đếm **xe nhìn thấy trong ảnh**, một ảnh có thể chứa nhiều loại; nhập 0 khi đã kiểm tra không có loại đó, để trống khi chưa kiểm tra. `reviewed=True` chỉ khi đã kiểm tra đầy đủ. Tổng xe qua ảnh là số lượt xuất hiện, không phải số xe duy nhất.

Notebook xuất `vehicle_review_template.csv` cho toàn bộ ảnh. Điền CSV, thêm vào Kaggle và đặt `VEHICLE_CSV` ở cell cấu hình rồi Run All. Thống kê dưới đây chỉ tính ảnh đã review, không ngoại suy phần chưa review. Metadata lớp được hiển thị ở mục 3; tên lớp cần được xác minh về ngữ nghĩa trước khi dùng cho báo cáo.


In [ ]:
vehicle_template=images[['relative_path','split']].copy()
count_cols=['n_motorcycles','n_cars','n_other_vehicles']
for col in count_cols: vehicle_template[col]=pd.NA
vehicle_template['reviewed']=False
vehicle_template.to_csv(OUT/'vehicle_review_template.csv',index=False,encoding='utf-8-sig')
vehicle_summary=None
vehicle_note='Chưa xác định số xe máy/ô tô từ annotation hiện có; cần nhãn loại xe hoặc kiểm đếm đã xác minh.'
if VEHICLE_CSV:
    v=pd.read_csv(VEHICLE_CSV)
    required=['relative_path',*count_cols,'reviewed']
    if not set(required).issubset(v.columns): raise ValueError('CSV thiếu cột: '+str(required))
    if v.relative_path.duplicated().any(): raise ValueError('CSV có relative_path trùng')
    unknown=set(v.relative_path)-set(images.relative_path)
    if unknown: raise ValueError('CSV chứa ảnh không thuộc dataset: '+str(sorted(unknown)[:5]))
    checked=v.reviewed.astype(str).str.strip().str.lower().isin(['true','1','yes'])
    vv=v[checked].copy()
    for col in count_cols:
        vv[col]=pd.to_numeric(vv[col],errors='raise')
        if vv[col].isna().any() or not (np.isfinite(vv[col]) & (vv[col]>=0) & (vv[col]%1==0)).all():
            raise ValueError('Ảnh reviewed phải có số đếm nguyên không âm: '+col)
    vv=vv[required].merge(images[['relative_path','split']],on='relative_path',validate='one_to_one')
    vehicle_summary=vv.groupby('split')[count_cols].sum()
    show_table('Số lượt xe trên ảnh đã review',vehicle_summary)
    vehicle_note=f'Đã kiểm đếm {len(vv)}/{len(images)} ảnh; xe máy={int(vv.n_motorcycles.sum())}, ô tô={int(vv.n_cars.sum())}, xe khác={int(vv.n_other_vehicles.sum())}. Đây là số lượt xuất hiện trong phần đã review.'
display(Markdown(vehicle_note))


## 11. Kết luận tự động và xuất báo cáo

Các con số được lấy từ kết quả chạy, không điền tay trước khi đọc dataset. Đề xuất xử lý giữ nguyên dữ liệu gốc: review nhãn thiếu/rỗng/lỗi; xác minh ý nghĩa lớp; xử lý ảnh trùng trước khi đánh giá; giữ cân bằng kích thước, độ sáng và lớp giữa các tập. Ảnh cùng nguồn/cùng video/cùng xe cần chia theo nhóm khi có metadata.

Chưa xác minh nguồn chụp, phân bố địa lý, thời gian, danh tính xe hay tính đại diện ngoài dataset. Không thể đánh giá chất lượng mô hình chỉ từ EDA.


In [ ]:
report_lines=[
    '# Báo cáo EDA – Vietnamese License Plate Detection',
    'Nguồn: https://www.kaggle.com/datasets/miahuynh04/vietnamese-license-plate-detection',
    'Thời điểm chạy (UTC): '+datetime.now(timezone.utc).isoformat(),
    f'- Dataset đọc từ: `{ROOT}`.',
    f'- Có **{len(images):,} sample ảnh**, {len(labels):,} file nhãn, {len(ann):,} dòng annotation.',
    f'- **{int(images.label_path.notna().sum()):,} ảnh có file nhãn**; **{int(images.has_valid_box.sum()):,} ảnh ghép duy nhất có ít nhất một box hợp lệ**.',
    f'- {int(images.ready_positive.sum()):,} ảnh dương tính đọc được, nhãn ghép duy nhất và không có dòng lỗi.',
    f'- {int((~images.readable).sum()):,} ảnh lỗi đọc; {int(images.label_status.eq("missing").sum()):,} ảnh thiếu nhãn; {int(labels.status.eq("empty").sum()):,} file nhãn rỗng.',
    f'- Có {int((~ann.valid).sum()):,} dòng nhãn lỗi/khác định dạng; {len(orphans):,} file nhãn mồ côi.',
    f'- Các class ID hợp lệ quan sát: {sorted(ann.loc[ann.valid,"class_id"].astype(int).unique().tolist())}; tên có tài liệu: {class_names or "chưa có"}.',
    f'- Trùng pixel: {len(pixel_dups):,} nhóm; {int(pixel_dups.n_splits.gt(1).sum()):,} nhóm xuất hiện ở nhiều split.',
    '- '+vehicle_note,
]
if len(readable):
    report_lines += [f'- Chiều rộng {int(readable.width.min())}–{int(readable.width.max())} px, chiều cao {int(readable.height.min())}–{int(readable.height.max())} px.',
                    f'- Kích thước trung vị: rộng {readable.width.median():g} px, cao {readable.height.median():g} px (hai trung vị riêng).',
                    '- Định dạng decode: '+', '.join(f'{k}: {v}' for k,v in readable.format.value_counts().items())+'.']
report_lines += ['', '## Theo split', split_summary.to_csv(),
    '## Giới hạn và việc cần làm',
    '- Sample là file ảnh; box là dòng annotation. Các số gốc chưa loại annotation/ảnh trùng.',
    '- Nhãn rỗng chưa chắc là ảnh âm tính; thiếu nhãn không chứng minh không có biển số.',
    '- dHash chỉ phát hiện ứng viên collision, chưa dò đầy đủ ảnh gần trùng.',
    '- Xác minh tên lớp và nguồn ảnh; review lỗi nhãn, ảnh nhỏ, ảnh tối và rò rỉ giữa split.',
    '- Không suy loại xe từ hình dạng biển số. Đếm xe cần nhãn loại xe hoặc kiểm đếm trực quan có coverage rõ ràng.']
report='\n'.join(report_lines)
(OUT/'report_vi.md').write_text(report,encoding='utf-8')
display(Markdown(report))
tables={'file_inventory':inventory,'image_metadata':images,'label_files':labels,
    'annotation_rows':ann,'valid_boxes':boxes,'split_summary':split_summary.reset_index(),
    'headline':headline,'image_size_statistics':size_stats.reset_index(),
    'class_summary':class_summary,'quality_summary':quality_summary,
    'orphan_labels':orphans,'duplicate_annotations':duplicate_annotations,
    'byte_duplicate_groups':byte_dups,'pixel_duplicate_groups':pixel_dups,
    'dhash_candidates':dhash_candidates}
if vehicle_summary is not None: tables['vehicle_counts_reviewed']=vehicle_summary.reset_index()
for name,table in tables.items(): table.to_csv(OUT/(name+'.csv'),index=False,encoding='utf-8-sig')
manifest={'source':'miahuynh04/vietnamese-license-plate-detection','root':str(ROOT),
    'seed':SEED,'utc':datetime.now(timezone.utc).isoformat(),
    'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
                'pillow':Image.__version__,'matplotlib':matplotlib.__version__},
    'class_names':class_names,'class_name_sources':name_sources,
    'sha256_file_manifest':'image_metadata.csv',
    'note':'Các thống kê chưa deduplicate; chỉ parser detection YOLO 5 cột.'}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
import shutil
archive=shutil.make_archive(str(OUT.parent/'eda_report_bundle'),'zip',OUT)
print('Báo cáo, CSV và biểu đồ:',OUT)
display(FileLink(str(OUT/'report_vi.md')))
display(FileLink(archive))
